In [1]:
import time
import shutil
import tempfile
from pathlib import Path

from pdfqa_rag.config import AppConfig, settings
from pdfqa_rag.store.factory import build_vector_store
from pdfqa_rag.pipeline.factory import build_document_ingest_pipeline

# Point at epyc's GPU services for larger batches — for local-only sidecars,
# just don't set these env vars (config defaults to localhost:8021/8031/8032).
import os
os.environ.setdefault("DOC_INTEL_SERVICE_URL", "http://192.168.0.11:8021")
os.environ.setdefault("DOC_INTEL_TIMEOUT_S", "90")
os.environ.setdefault("MM_EMBED_EMBED_SERVER_URL", "http://192.168.0.11:8031")
os.environ.setdefault("MM_EMBED_RERANK_SERVER_URL", "http://192.168.0.11:8032")
os.environ.setdefault("MM_EMBED_TIMEOUT_S", "60")

cfg = AppConfig()

# File-based, real vector search (LanceDB), persists across kernel restarts —
# swap backend="pgvector" later with no other code changes.
store = build_vector_store(
    cfg.store.__class__(backend="lancedb", lancedb_path="data/lancedb"),
    dimensions=2048,
)
pipeline = build_document_ingest_pipeline(cfg, store)

# Curated small (3-17 page) files instead of the first 5 alphabetically —
# the alphabetical set includes a 154-page file that times out on CPU-bound
# rasterization regardless of timeout, and one file with a real active-content
# PDF finding that correctly stays blocked by the security scan.
src_dir = settings.ROOT_DIR / "data/pdfQA-Benchmark/real-pdfQA/01.2_Input_Files_PDF/NaturalQuestions"
SMALL_FILES = [
    "1990 AFL Grand Final.pdf",
    "1972 Miami Dolphins season.pdf",
    "2018 Australian Grand Prix.pdf",
    "1998 Winter Olympics.pdf",
    "2016 ICC Women's World Twenty20.pdf",
]
dataset_dir = Path(tempfile.mkdtemp())
for f in SMALL_FILES:
    shutil.copy(src_dir / f, dataset_dir / f)
print(dataset_dir, len(list(dataset_dir.glob("*.pdf"))), "pdfs staged")

t0 = time.time()
stats = await pipeline.ingest_dataset(
    dataset_dir,
    collection="nq-sample",
    limit=5,
    # concurrency=1 by default — document-intelligence-gpu appears to
    # serialize extraction internally; see ingest_dataset's own docstring.
)
print(f"stats: {stats}  ({time.time()-t0:.1f}s)")

await pipeline.aclose()

/tmp/tmp31c9g17y 5 pdfs staged


Skipping image img-p5-0 in 1998 Winter Olympics.pdf: llama-embed sidecar request failed (http://192.168.0.11:8031): Client error '400 Bad Request' for url 'http://192.168.0.11:8031/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
Skipping image img-p7-0 in 1998 Winter Olympics.pdf: llama-embed sidecar request failed (http://192.168.0.11:8031): Client error '400 Bad Request' for url 'http://192.168.0.11:8031/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
Batch embed failed for 2016 ICC Women's World Twenty20.pdf (llama-embed sidecar batch request failed (http://192.168.0.11:8031): Client error '400 Bad Request' for url 'http://192.168.0.11:8031/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400) — falling back to per-chunk
Skipping chunk 12 in 2016 ICC Women's World Twenty20.pdf: llama-embed sidecar request failed (http://192.168.0.1

stats: {'files': 5, 'failed': 0, 'text_docs': 88, 'image_docs': 23}  (61.4s)


In [ ]:
from substrate.runtimes.embedding_reranker.service.embedding import EmbeddingReranker

from pdfqa_rag.config import AppConfig, StoreConfig
from pdfqa_rag.store.factory import build_vector_store

cfg = AppConfig()
er = EmbeddingReranker(
    embed_server_url=cfg.mm_embed.embed_server_url, 
    rerank_server_url=cfg.mm_embed.rerank_server_url
)
store = build_vector_store(
    StoreConfig(backend="lancedb", lancedb_path="data/lancedb"),
    dimensions=2048,
)

query_vec = await er.embed_text("get me some info about france")
results = await store.search(query_vec, collection="nq-sample", limit=5)
for r in results:
    print(round(r.score, 3), r.metadata.get("source"), r.to_text()[:500])

await er.aclose()

0.627 2018 Australian Grand Prix.pdf <div style="text-align: center;"><html><body>
0.6 1998 Winter Olympics.pdf <div style="text-align: center;">All dates are in Japan Standard Time (UTC+9)</div>



<div style="text-align: center;"><html><body>
0.546 1998 Winter Olympics.pdf 1. "The Olympic Winter Games Factsheet" (http://www.olympic.org/Documents/Reference_docu ments Factsheets/The_Olympic_Winter_Games.pdf) (PDF). International Olympic Committee.Retrieved 5 August 2012. 2. "French and English are the official languages for the Olympic Games.", [1] (http://www.olympi C.org/fr/).(..)

3. Jordan, Mary; Sullivan, Kevin (21 January 1999), "Nagano Burned Documents Tracing '98Olympics Bid" (https://www.washingtonpost.com/wp-srv/digest/daily/jan99/nagano21.htm)Washington P
0.544 1998 Winter Olympics.pdf 1. "The Olympic Winter Games Factsheet" (http://www.olympic.org/Documents/Reference_docu ments Factsheets/The_Olympic_Winter_Games.pdf) (PDF). International Olympic Committee.Retrieved 5 Augus